# Single-power campaign — analysis template

**What this is.** A copy-me notebook for a *single-power* run: the pump power is fixed and we study **stability over time** by treating each acquisition **chunk** as a point. (To instead compare different *setups* at the same power, load each setup as its own run and pass them to `Campaign(...)` directly — see the last optional cell.)

All the analysis lives in `src/`; this notebook just configures and calls it, then exports a report under `results/`.

In [ ]:
# --- bootstrap: make `src` importable and run from the project root ---
import sys, os
from pathlib import Path
ROOT = Path.cwd()
if not (ROOT / 'src').is_dir():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
os.chdir(ROOT)

from src import Campaign, GridVisualizer, CampaignReport, HBTMeasurement, config
config.apply_style(usetex=True)

## 1. Configure & load
Point `RUN_DIR` at the acquisition folder; every `*chunk*.pkl` becomes one time point.

In [ ]:
RUN_DIR   = 'data/Jun17/SinglePower_GaAs100_700-40_P1'   # <-- edit me
HARMONICS = (3, 5)
TAU_INT   = 4.0

camp = Campaign.from_chunks(RUN_DIR, harmonics=HARMONICS, tau_in_ns=TAU_INT)
print(camp.runs[0].acquisition_summary())
camp.summary_table()

## 2. Stability vs time (chunk)
g²(0), R, singles and the normalised excess across chunks — flat curves mean a stable measurement.

In [ ]:
camp.plot_overview();

## 3. Coherence & sweeps (overlaying chunks)
Useful to confirm peak shape and to pick `TAU_INT` on the plateau.

In [ ]:
gv = GridVisualizer(camp.runs, labels=camp.labels, comparison_variable='Chunk (time)')
gv.plot_coherence(xlim=8.0, integration_window_ns=TAU_INT)
gv.plot_g2(methods=['delay'], tau_min=0.3, tau_max=25, step=1.0);

## 4. Export the report

In [ ]:
CampaignReport(camp).export(matrices=True)

## (optional) Compare setups at the same power
Load one representative run per setup and build a campaign with a custom control axis.

In [ ]:
# from src import Control
# setups = {
#     '700-40 P1':   'data/Jun17/SinglePower_GaAs100_700-40_P1/MERGED',
#     '700-10 P1':   'data/Jun17/SinglePower_GaAs100_700-10_P1_MaxGain/MERGED',
# }
# runs, labels = [], []
# for name, d in setups.items():
#     f = sorted(Path(d).glob('*MERGED*.pkl'))[0]
#     runs.append(HBTMeasurement(f)); labels.append(name)
# ctrl = Control.custom(lambda r, i: i, label='Setup', fmt=lambda v: labels[int(v)])
# comp = Campaign(runs, control=ctrl, harmonics=HARMONICS, tau_in_ns=TAU_INT, name='setup_compare')
# comp.plot_overview();